# Faster R-CNN (ResNet-50 + FPN) — Training Pipeline

Fine-tuning pipeline for GUI element detection.

**Features:**
- Reproducibility (fixed seeds)
- Mixed precision (AMP) for faster training
- Unified COCO evaluation (mAP@50 and mAP@50:95 in one call)
- Fallback when no detections are produced
- Functional early stopping
- Modular and readable code

In [ ]:
import os
import json
import random
import numpy as np
import torch
import torchvision
from PIL import Image
from torch.utils.data import DataLoader
from torchvision.datasets import CocoDetection
from torchvision.transforms import functional as F
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.amp import GradScaler

from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from tqdm import tqdm
import matplotlib.pyplot as plt

## Global Configuration

In [ ]:
DATASET_ROOT = "../../data/unified/combined"

TRAIN_IMAGES_PATH = f"{DATASET_ROOT}/train/images"
TRAIN_ANNOT_PATH  = f"{DATASET_ROOT}/train/annotations/train.json"
VAL_IMAGES_PATH   = f"{DATASET_ROOT}/val/images"
VAL_ANNOT_PATH    = f"{DATASET_ROOT}/val/annotations/val.json"

NUM_CLASSES      = 13   # 12 unified classes + 1 background
BATCH_SIZE       = 8
NUM_EPOCHS       = 30
INITIAL_LR       = 0.002
MOMENTUM         = 0.9
WEIGHT_DECAY     = 0.0005
SCHED_PATIENCE   = 7
ES_PATIENCE      = 10   # early stopping: para após 10 epochs sem melhoria
LR_FACTOR        = 0.5
SEED             = 42
WARMUP_EPOCHS    = 5
NUM_WORKERS      = 0    # 0 no Windows/Jupyter (evita problemas de multiprocessing)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")

## Reproducibility

In [ ]:
def fix_seeds(seed: int = 42):
    """Fix all seeds for reproducible results."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

## Transforms / Augmentations

In [ ]:
class TrainTransform:
    """
    Training augmentations:
      - Colour jitter (brightness, contrast, saturation, hue)
      - Horizontal flip (with bounding box adjustment)
      - Random resize (multi-scale training)
    """

    def __init__(self, flip_prob=0.5, jitter_params=(0.3, 0.3, 0.2, 0.1),
                 scales=(480, 512, 544, 576, 608, 640)):
        self.flip_prob = flip_prob
        self.jitter = torchvision.transforms.ColorJitter(*jitter_params)
        self.scales = scales

    def __call__(self, image, annotations):
        image = self.jitter(image)

        if random.random() < self.flip_prob:
            image = F.hflip(image)
            width, _ = image.size
            for obj in annotations:
                x, y, w, h = obj["bbox"]
                obj["bbox"][0] = width - x - w

        if self.scales:
            target_scale = random.choice(self.scales)
            orig_w, orig_h = image.size
            ratio = target_scale / min(orig_w, orig_h)
            new_w = int(orig_w * ratio)
            new_h = int(orig_h * ratio)
            image = image.resize((new_w, new_h), Image.BILINEAR)

            for obj in annotations:
                obj["bbox"][0] *= ratio
                obj["bbox"][1] *= ratio
                obj["bbox"][2] *= ratio
                obj["bbox"][3] *= ratio

        image = F.to_tensor(image)
        return image, annotations


class ValTransform:
    """Minimal transform for validation: only converts to tensor."""

    def __call__(self, image, annotations):
        return F.to_tensor(image), annotations

## Dataset and DataLoaders

In [ ]:
def create_dataset(images_path, annot_path, transforms):
    """Create a COCO dataset with the given transforms."""
    return CocoDetection(
        root=images_path,
        annFile=annot_path,
        transforms=transforms,
    )


def collate_fn(batch):
    """
    Custom collate function.
    Needed because each image can have a different number of objects.
    Returns a tuple of (images_list, annotations_list).
    """
    return tuple(zip(*batch))


def create_dataloaders(batch_size):
    """Create training and validation DataLoaders."""
    train_ds = create_dataset(
        TRAIN_IMAGES_PATH, TRAIN_ANNOT_PATH,
        transforms=TrainTransform(),
    )
    val_ds = create_dataset(
        VAL_IMAGES_PATH, VAL_ANNOT_PATH,
        transforms=ValTransform(),
    )

    train_loader = DataLoader(
        train_ds, batch_size=batch_size,
        shuffle=True, collate_fn=collate_fn,
        num_workers=NUM_WORKERS, pin_memory=False,
    )
    val_loader = DataLoader(
        val_ds, batch_size=batch_size,
        shuffle=False, collate_fn=collate_fn,
        num_workers=NUM_WORKERS, pin_memory=False,
    )

    return train_loader, val_loader

## Model

In [ ]:
def create_model(num_classes):
    """
    Load pre-trained Faster R-CNN (ResNet-50 + FPN) and replace
    the classification head for our number of classes.
    """
    model = torchvision.models.detection.fasterrcnn_resnet50_fpn(
        weights="DEFAULT"
    )

    num_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(num_features, num_classes)

    return model

## Training Loop (1 Epoch)

In [ ]:
def train_one_epoch(model, optimiser, loader, device, epoch, total_epochs, scaler):
    model.train()

    loss_sums = {"box": 0.0, "cls": 0.0, "obj": 0.0, "rpn_box": 0.0}
    num_batches = 0

    print(f"  Epoch {epoch}/{total_epochs} — loading first batch...")
    progress = tqdm(loader, desc=f"Epoch {epoch}/{total_epochs}", leave=True)

    for images, annotations in progress:
        images = [img.to(device) for img in images]

        valid_targets = []
        valid_images = []

        for img, anns in zip(images, annotations):
            boxes, labels = [], []
            for obj in anns:
                x, y, w, h = obj["bbox"]
                if w > 0 and h > 0:
                    boxes.append([x, y, x + w, y + h])
                    labels.append(obj["category_id"])

            if boxes:
                valid_targets.append({
                    "boxes":  torch.tensor(boxes, dtype=torch.float32, device=device),
                    "labels": torch.tensor(labels, dtype=torch.int64, device=device),
                })
                valid_images.append(img)

        if not valid_targets:
            continue

        with torch.amp.autocast("cuda", enabled=(device.type == "cuda")):
            loss_dict = model(valid_images, valid_targets)
            loss_total = sum(loss_dict.values())

        optimiser.zero_grad()
        scaler.scale(loss_total).backward()
        scaler.unscale_(optimiser)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=10.0)
        scaler.step(optimiser)
        scaler.update()

        loss_sums["box"]     += loss_dict["loss_box_reg"].item()
        loss_sums["cls"]     += loss_dict["loss_classifier"].item()
        loss_sums["obj"]     += loss_dict["loss_objectness"].item()
        loss_sums["rpn_box"] += loss_dict["loss_rpn_box_reg"].item()
        num_batches += 1

        progress.set_postfix(
            box=f"{loss_sums['box'] / num_batches:.3f}",
            cls=f"{loss_sums['cls'] / num_batches:.3f}",
            rpn=f"{(loss_sums['obj'] + loss_sums['rpn_box']) / num_batches:.3f}",
        )

    avg_box = loss_sums["box"] / max(num_batches, 1)
    avg_cls = loss_sums["cls"] / max(num_batches, 1)
    avg_rpn = (loss_sums["obj"] + loss_sums["rpn_box"]) / max(num_batches, 1)

    return avg_box, avg_cls, avg_rpn

## Evaluation (COCO Metrics)

In [ ]:
def evaluate(model, loader, device, annot_path):
    """
    Evaluates the model on the validation set using the COCO API.

    Returns a dictionary with:
      - map50:    mAP @ IoU=0.50
      - recall50: recall @ IoU=0.50
      - map:      mAP @ IoU=0.50:0.95 (standard COCO metric)
    """
    model.eval()
    coco_gt = COCO(annot_path)
    results = []

    with torch.no_grad():
        for images, annotations in loader:
            images = [img.to(device) for img in images]
            outputs = model(images)

            for anns, output in zip(annotations, outputs):
                if not anns:
                    continue

                image_id = int(anns[0]["image_id"])
                boxes  = output["boxes"].cpu().numpy()
                scores = output["scores"].cpu().numpy()
                labels = output["labels"].cpu().numpy()

                for box, score, label in zip(boxes, scores, labels):
                    x1, y1, x2, y2 = box
                    results.append({
                        "image_id":    image_id,
                        "category_id": int(label),
                        "bbox":        [float(x1), float(y1), float(x2 - x1), float(y2 - y1)],
                        "score":       float(score),
                    })

    if not results:
        print("No detections produced — returning zero metrics.")
        return {"map50": 0.0, "recall50": 0.0, "map": 0.0}

    coco_dt = coco_gt.loadRes(results)
    evaluator = COCOeval(coco_gt, coco_dt, iouType="bbox")
    evaluator.evaluate()
    evaluator.accumulate()

    precision = evaluator.eval["precision"]
    recall   = evaluator.eval["recall"]

    prec_50 = precision[0]
    map50 = float(np.mean(prec_50[prec_50 > -1]))

    rec_50 = recall[0]
    recall50 = float(np.mean(rec_50[rec_50 > -1]))

    map_full = float(np.mean(precision[precision > -1]))

    return {"map50": map50, "recall50": recall50, "map": map_full}

## LR Warmup and Training Plots

In [ ]:
def adjust_lr_warmup(optimiser, epoch, warmup_epochs, target_lr):
    """
    Linear warmup: during the first epochs, the LR rises gradually
    from a very low value to the target LR. This stabilises early
    training and prevents gradient explosions at the start.
    """
    if epoch <= warmup_epochs:
        lr = target_lr * (epoch / warmup_epochs)
        for param_group in optimiser.param_groups:
            param_group["lr"] = lr
        return lr
    return optimiser.param_groups[0]["lr"]


def generate_plots(history, output_path="training_and_metrics.png"):
    """
    Generate a 2×3 panel with training loss curves and evaluation metrics.
    """
    epochs = list(range(1, len(history["train_box_loss"]) + 1))
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))

    loss_configs = [
        ("train_box_loss", "Train — Box Loss"),
        ("train_cls_loss", "Train — Classification Loss"),
        ("train_rpn_loss", "Train — RPN Loss"),
    ]
    for i, (key, title) in enumerate(loss_configs):
        axes[0, i].plot(epochs, history[key], "-o", markersize=3)
        axes[0, i].set_title(title)
        axes[0, i].set_xlabel("Epoch")
        axes[0, i].set_ylabel("Loss")
        axes[0, i].grid(True, alpha=0.3)

    metric_configs = [
        ("map50",    "mAP @ IoU=0.50"),
        ("recall50", "Recall @ IoU=0.50"),
        ("map",      "mAP @ IoU=0.50:0.95"),
    ]
    for i, (key, title) in enumerate(metric_configs):
        axes[1, i].plot(epochs, history[key], "-o", markersize=3, color="green")
        axes[1, i].set_title(title)
        axes[1, i].set_xlabel("Epoch")
        axes[1, i].set_ylabel("Value")
        axes[1, i].grid(True, alpha=0.3)

    fig.tight_layout()
    fig.savefig(output_path, dpi=150)
    plt.close(fig)
    print(f"Plot saved to: {output_path}")

## Main Training Loop

In [ ]:
fix_seeds(SEED)
print(f"Device: {DEVICE}")

train_loader, val_loader = create_dataloaders(BATCH_SIZE)
print(f"Train: {len(train_loader.dataset)} images | "
      f"Val: {len(val_loader.dataset)} images")

model = create_model(NUM_CLASSES).to(DEVICE)

optimiser = torch.optim.SGD(
    model.parameters(),
    lr=INITIAL_LR,
    momentum=MOMENTUM,
    weight_decay=WEIGHT_DECAY,
)
scheduler = ReduceLROnPlateau(
    optimiser,
    mode="max",
    factor=LR_FACTOR,
    patience=SCHED_PATIENCE,
)

scaler = GradScaler("cuda", enabled=(DEVICE.type == "cuda"))

history = {
    "train_box_loss": [], "train_cls_loss": [], "train_rpn_loss": [],
    "map50": [], "recall50": [], "map": [],
}

best_map50 = 0.0
epochs_no_improvement = 0

for epoch in range(1, NUM_EPOCHS + 1):

    current_lr = adjust_lr_warmup(optimiser, epoch, WARMUP_EPOCHS, INITIAL_LR)

    box_loss, cls_loss, rpn_loss = train_one_epoch(
        model, optimiser, train_loader,
        DEVICE, epoch, NUM_EPOCHS, scaler,
    )

    metrics = evaluate(model, val_loader, DEVICE, VAL_ANNOT_PATH)

    history["train_box_loss"].append(box_loss)
    history["train_cls_loss"].append(cls_loss)
    history["train_rpn_loss"].append(rpn_loss)
    history["map50"].append(metrics["map50"])
    history["recall50"].append(metrics["recall50"])
    history["map"].append(metrics["map"])

    current_lr = optimiser.param_groups[0]["lr"]
    print(
        f"\nEpoch {epoch}/{NUM_EPOCHS} — "
        f"LR: {current_lr:.6f} | "
        f"mAP50: {metrics['map50']:.4f} | "
        f"Recall50: {metrics['recall50']:.4f} | "
        f"mAP50-95: {metrics['map']:.4f}"
    )

    if epoch > WARMUP_EPOCHS:
        scheduler.step(metrics["map50"])

    if metrics["map50"] > best_map50:
        best_map50 = metrics["map50"]
        torch.save(model.state_dict(), "best_fasterrcnn.pth")
        epochs_no_improvement = 0
        print(f"New best model saved (mAP50 = {best_map50:.4f})")
    else:
        epochs_no_improvement += 1
        print(f"No improvement ({epochs_no_improvement}/{ES_PATIENCE})")

    if epochs_no_improvement >= ES_PATIENCE:
        print(f"\n Training stopped — no improvement for {ES_PATIENCE} epochs.")
        break

generate_plots(history)
print("\n Training complete!")

In [ ]:
# Diagnóstico: testa se consegue carregar as primeiras 3 imagens
import time

print("A testar carregamento de imagens...")
test_ds = create_dataset(TRAIN_IMAGES_PATH, TRAIN_ANNOT_PATH, transforms=ValTransform())

for i in range(3):
    t = time.time()
    print(f"  A carregar imagem {i}...", end=" ", flush=True)
    try:
        img, ann = test_ds[i]
        print(f"OK — shape: {img.shape}, anotações: {len(ann)}, tempo: {time.time()-t:.2f}s")
    except Exception as e:
        print(f"ERRO: {e}")

print("Diagnóstico concluído.")